# 03 — Chia dữ liệu 70/15/15 và tiền xử lý không rò rỉ dữ liệu

**RT-IoT2022 | 12 lớp gốc | Baseline multiclass**

Notebook đọc CSV sau tiền xử lý trước chia; chia phân tầng, fit toàn bộ biến đổi trên Train, rồi transform Validation/Test. Không cân bằng lớp trước chia, không chọn đặc trưng bằng thông tin từ Validation/Test.

## 1. Cấu hình và kiểm tra dữ liệu đầu vào

In [ ]:
from pathlib import Path
import json
import numpy as np
import pandas as pd
import joblib
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer

SEED = 42
TARGET = "Attack_type"
# Notebook nằm trong notebooks/ hoặc chạy trực tiếp cạnh file CSV đều được.
# T? t?m ??u v?o trong data/experiments/baseline/processed.
CANDIDATES = [root / "data" / "experiments" / "baseline" / "processed" / "RT_IOT2022_pre_split_12_classes.csv" for root in [Path.cwd(), *Path.cwd().parents]]
INPUT_PATH = next((p for p in CANDIDATES if p.is_file()), None)
if INPUT_PATH is None:
    raise FileNotFoundError("Ch?a c? d? li?u processed. H?y ch?y 02_preprocess.ipynb tr??c.")
PROJECT_ROOT = next(p.parent.parent.parent.parent.parent for p in CANDIDATES if p == INPUT_PATH)
OUTPUT_DIR = PROJECT_ROOT / "data" / "experiments" / "baseline" / "splits"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
df = pd.read_csv(INPUT_PATH, low_memory=False)
assert TARGET in df.columns
assert df[TARGET].nunique() == 12, "Đầu vào phải có đúng 12 lớp."
assert not df[TARGET].isna().any()
print("Input:", INPUT_PATH, df.shape)
print(df[TARGET].value_counts().to_string())

## 2. Stratified split 70% / 15% / 15%
Giữ nguyên 12 nhãn. Xuất `train_raw.csv`, `val_raw.csv`, `test_raw.csv` để lưu dữ liệu trước scaling/encoding.

In [ ]:
# Hai lần chia để đạt 70/15/15 và giữ tỷ lệ từng lớp.
train_df, temp_df = train_test_split(
    df, test_size=0.30, stratify=df[TARGET], random_state=SEED
)
val_df, test_df = train_test_split(
    temp_df, test_size=0.50, stratify=temp_df[TARGET], random_state=SEED
)
# Giữ index gốc trong bộ nhớ để kiểm tra tập giao nhau, rồi reset trước khi xuất.
assert not set(train_df.index) & set(val_df.index)
assert not set(train_df.index) & set(test_df.index)
assert not set(val_df.index) & set(test_df.index)
assert len(train_df) + len(val_df) + len(test_df) == len(df)
for name, part in [("train",train_df),("val",val_df),("test",test_df)]:
    part.reset_index(drop=True).to_csv(OUTPUT_DIR/f"{name}_raw.csv",index=False)
print("Sizes:",len(train_df),len(val_df),len(test_df))

## 3. Fit trên Train; transform ba tập
Numeric: median imputation + StandardScaler. Categorical: most-frequent imputation + OneHotEncoder(handle_unknown=ignore). **Không bao giờ fit trên Validation hoặc Test.** Đầu ra có cùng tên và thứ tự cột.

In [ ]:
# Không đưa nhãn hoặc bất kỳ cột suy ra từ nhãn vào đặc trưng.
X_train = train_df.drop(columns=[TARGET])
X_val = val_df.drop(columns=[TARGET])
X_test = test_df.drop(columns=[TARGET])
y_train = train_df[TARGET].reset_index(drop=True)
y_val = val_df[TARGET].reset_index(drop=True)
y_test = test_df[TARGET].reset_index(drop=True)
num_cols = X_train.select_dtypes(include=["number"]).columns.tolist()
cat_cols = X_train.select_dtypes(exclude=["number"]).columns.tolist()
assert set(num_cols).isdisjoint(cat_cols)
assert len(num_cols)+len(cat_cols)==X_train.shape[1]
print("Numeric:",len(num_cols),"Categorical:",cat_cols)

numeric_pipe = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler()),
])
categorical_pipe = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore", sparse_output=False, dtype=np.float32)),
])
preprocessor = ColumnTransformer([
    ("num", numeric_pipe, num_cols),
    ("cat", categorical_pipe, cat_cols),
], remainder="drop", sparse_threshold=0)
# CHỈ FIT TRÊN TRAIN
X_train_t = preprocessor.fit_transform(X_train)
X_val_t = preprocessor.transform(X_val)
X_test_t = preprocessor.transform(X_test)
feature_names = preprocessor.get_feature_names_out().tolist()
assert TARGET not in feature_names
assert len(set(feature_names))==len(feature_names)

def export_transformed(matrix, labels, name):
    frame = pd.DataFrame(matrix, columns=feature_names).astype(np.float32)
    frame[TARGET] = labels.to_numpy()
    assert not frame.isna().any().any(), f"NaN: {name}"
    assert np.isfinite(frame.drop(columns=[TARGET]).to_numpy()).all(), f"Inf: {name}"
    frame.to_csv(OUTPUT_DIR/f"{name}_processed.csv",index=False, float_format="%.7g")
    return frame
train_processed=export_transformed(X_train_t,y_train,"train")
val_processed=export_transformed(X_val_t,y_val,"val")
test_processed=export_transformed(X_test_t,y_test,"test")
joblib.dump(preprocessor, OUTPUT_DIR/"preprocessor_train_only.joblib")
print("Transformed shape:",train_processed.shape,val_processed.shape,test_processed.shape)

## 4. Kiểm tra tính toàn vẹn, phân bố lớp và xuất metadata
Các file `*_processed.csv` chứa đặc trưng số đã biến đổi và nhãn gốc `Attack_type`. Giữ lại `preprocessor_train_only.joblib` để suy luận trên dữ liệu mới.

**Giới hạn:** Split ngẫu nhiên có phân tầng không bảo đảm độc lập giữa phiên mạng/thiết bị/thời gian. Nếu có group/session ID đáng tin cậy, cần thử group-based split. Một số biến thống kê có thể biểu hiện đặc tính đợt thu thập; cần kiểm tra khi đánh giá.

In [ ]:
# Phân bố nhãn trước/sau chia, cả số lượng và phần trăm.
counts = pd.concat({
    "all":df[TARGET].value_counts(),
    "train":train_df[TARGET].value_counts(),
    "val":val_df[TARGET].value_counts(),
    "test":test_df[TARGET].value_counts(),
},axis=1).fillna(0).astype(int)
for name,part in [("all",df),("train",train_df),("val",val_df),("test",test_df)]:
    counts[name+"_pct"] = (counts[name]/len(part)*100).round(4)
counts.index.name=TARGET
counts.to_csv(OUTPUT_DIR/"class_distribution.csv")
assert (counts[["train","val","test"]].sum(axis=1)==counts["all"]).all()
assert all(part[TARGET].nunique()==12 for part in [train_df,val_df,test_df])
metadata={
    "input_file":INPUT_PATH.name,"random_state":SEED,"target":TARGET,
    "split_ratio":{"train":0.70,"val":0.15,"test":0.15},
    "rows":{"all":len(df),"train":len(train_df),"val":len(val_df),"test":len(test_df)},
    "features_before":len(num_cols)+len(cat_cols),"features_after":len(feature_names),
    "numeric_columns":num_cols,"categorical_columns":cat_cols,
    "feature_names_after":feature_names,
    "classes":sorted(df[TARGET].unique().tolist()),
    "leakage_control":"Fit imputer, scaler, one-hot encoder on train only; val/test transform only; no resampling.",
    "warning":"Random stratified split does not rule out session/device/temporal leakage; audit groups if available."
}
(OUTPUT_DIR/"split_metadata.json").write_text(json.dumps(metadata,ensure_ascii=False,indent=2),encoding="utf-8")
print(counts.to_string())
print("Exported:",[p.name for p in OUTPUT_DIR.iterdir()])